# Universal Precision Runtime (UPR) — Notebook 03
## Level 4 (Variable Precision Sweep) & Level 5 (WikiText Perplexity Audit)

---

### Objective
1. Enforce deterministic evaluation (`upr.set_seed(42)`) for every precision level — Fix 10.
2. Reconstruct models across precisions (16, 14, 12, 10, 8, 6, 4, 2 bits) from a single BitPlane checkpoint.
3. Track isolated timing (`IsolatedTimer`) — Fix 6.
4. Record memory statistics (`MemoryProfiler`) to `results/memory.csv` — Fix 7.
5. Export per-tensor reconstruction metrics to `results/reconstruction.csv` — Fix 5.
6. Save unified schema JSON for each precision in `results/` with experiment metadata — Fix 11, 12.

In [ ]:
import os
import sys
import gc
import json
import importlib
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

# Set Hugging Face Token safely from environment or Colab secrets
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    if not HF_TOKEN:
        HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.chdir(DRIVE_DIR)
except ImportError:
    pass

WORK_DIR = os.getcwd()
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import upr
importlib.reload(upr)
upr.set_seed(42)

MODEL_ID = 'Qwen/Qwen3.5-0.8B'
BITPLANE_DIR = 'models/bitplane_qwen'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Fix 9 — Use WikiText-2 with identical tokenizer/prompt/generation settings every run
print('Loading WikiText-2 dataset...')
try:
    test_dataset = load_dataset('salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
except Exception:
    test_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', trust_remote_code=True)

text_samples = [t for t in test_dataset['text'] if len(t.strip()) > 100][:32]
encodings = tokenizer('\n\n'.join(text_samples), return_tensors='pt')
seq_len = 512
input_ids = encodings.input_ids[:, :seq_len * 4].to(DEVICE)

# Identical prompt and generation settings for every precision — Fix 9
EVAL_PROMPT = "Universal Precision Runtime provides dynamic multi-precision execution."
GEN_PROMPT = "The key advantage of a single bit-plane representation is"
GEN_MAX_NEW_TOKENS = 40

def evaluate_perplexity(model, input_ids, seq_len=512):
    model.eval()
    nlls = []
    total_len = input_ids.size(1)
    end_loc = 0
    for i in range(0, total_len, seq_len):
        end_loc = min(i + seq_len, total_len)
        if end_loc - i < 64:
            continue
        trg_len = end_loc - i
        chunk_ids = input_ids[:, i:end_loc]
        target_ids = chunk_ids.clone()
        with torch.no_grad():
            try:
                outputs = model(chunk_ids, labels=target_ids)
                loss = outputs.loss
                if torch.isnan(loss) or torch.isinf(loss):
                    return 9999.0
                nlls.append(loss * trg_len)
            except Exception:
                return 9999.0
    if not nlls or end_loc == 0:
        return 9999.0
    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    val = float(ppl.item())
    return val if not (torch.isnan(ppl) or torch.isinf(ppl)) else 9999.0

### Step 2: FP16 Baseline Evaluation

In [ ]:
print(f'Evaluating FP16 Baseline Model on {DEVICE}...')
orig_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
orig_model.eval()

prompt_inputs = tokenizer(EVAL_PROMPT, return_tensors='pt').to(DEVICE)
gen_inputs = tokenizer(GEN_PROMPT, return_tensors='pt').to(DEVICE)

with torch.no_grad():
    orig_logits = orig_model(**prompt_inputs).logits.detach().cpu()
    orig_output_ids = orig_model.generate(**gen_inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False).detach().cpu()

ppl_baseline = evaluate_perplexity(orig_model, input_ids, seq_len=seq_len)
orig_state_dict = orig_model.state_dict()

del orig_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'FP16 BASELINE RESULT | Perplexity (PPL): {ppl_baseline:.4f}')

### Step 3: Variable Precision Sweep (16, 14, 12, 10, 8, 6, 4, 2 Bits)

In [ ]:
precisions = [16, 14, 12, 10, 8, 6, 4, 2]
sweep_results = []
os.makedirs('results', exist_ok=True)
config = AutoConfig.from_pretrained(MODEL_ID)
mem_profiler = upr.MemoryProfiler()

print(f"{'Bits':<6} | {'Recon(s)':<10} | {'Init(s)':<8} | {'CosSim':<12} | {'Top-1%':<10} | {'PPL'}")
print("-" * 72)

for bits in precisions:
    upr.set_seed(42)  # Fix 10: identical seed per precision
    timer = upr.IsolatedTimer()

    # Fix 6 — Isolated Reconstruction Time
    timer.start("reconstruction")
    recon_state_dict = upr.BitPlaneModel.load_reconstructed_state_dict(
        bitplane_directory=BITPLANE_DIR,
        bits=bits,
        device='cpu',
        export_reconstruction_csv=(bits in [16, 8, 4]),  # Fix 5 — only for key levels
        original_state_dict=orig_state_dict,
        csv_output_path="results/reconstruction.csv"
    )
    t_recon = timer.stop("reconstruction")

    # Fix 6 — Isolated Model Init Time
    timer.start("model_init")
    recon_model = AutoModelForCausalLM.from_config(config, torch_dtype=torch.float16)
    recon_model.load_state_dict(recon_state_dict, strict=True)
    del recon_state_dict
    gc.collect()
    recon_model = recon_model.to(DEVICE)
    recon_model.eval()
    t_init = timer.stop("model_init")

    # Fix 6 — Isolated Forward Pass Time
    timer.start("forward_pass")
    try:
        with torch.no_grad():
            recon_logits = recon_model(**prompt_inputs).logits.detach().cpu()
        cos_sim = upr.compute_cosine_similarity(orig_logits, recon_logits)
        kl_div = upr.compute_kl_divergence(orig_logits, recon_logits)
    except Exception:
        cos_sim = 0.0
        kl_div = 0.0
    t_forward = timer.stop("forward_pass")

    # Fix 6 — Isolated Generation Time
    timer.start("generation")
    try:
        with torch.no_grad():
            recon_output_ids = recon_model.generate(**gen_inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False).detach().cpu()
        token_matches = (orig_output_ids == recon_output_ids).sum().item()
        token_acc = (token_matches / orig_output_ids.numel()) * 100.0
    except Exception:
        token_acc = 0.0
    t_gen = timer.stop("generation")

    ppl = evaluate_perplexity(recon_model, input_ids, seq_len=seq_len)

    # Fix 7 — Memory snapshot
    mem_stats = mem_profiler.record_memory_snapshot(bits, BITPLANE_DIR, "results/memory.csv")

    del recon_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Fix 11, 12 — Unified schema with metadata
    res_item = {
        "precision_bits": bits,
        "dataset": "wikitext-2-raw-v1",
        "seed": 42,
        "timing_sec": {
            "reconstruction": round(t_recon, 4),
            "model_init": round(t_init, 4),
            "forward_pass": round(t_forward, 4),
            "generation": round(t_gen, 4)
        },
        "memory_stats": mem_stats,
        "logit_cosine_similarity": cos_sim,
        "logit_kl_divergence": kl_div,
        "top1_token_accuracy_pct": token_acc,
        "perplexity": ppl,
        "metadata": upr.collect_experiment_metadata(precision_bits=bits)
    }
    sweep_results.append(res_item)

    with open(f'results/{bits}bit.json', 'w') as f:
        json.dump(res_item, f, indent=2)

    print(f"{bits:<6} | {t_recon:<10.3f} | {t_init:<8.3f} | {cos_sim:<12.6f} | {token_acc:<9.2f}% | {ppl:.4f}")

### Step 4: Export Unified Summary

In [ ]:
summary_data = {
    'baseline_model': MODEL_ID,
    'baseline_perplexity': ppl_baseline,
    'precision_bits': 16,
    'dataset': 'wikitext-2-raw-v1',
    'seed': 42,
    'precision_sweep': sweep_results
}

with open('results/variable_precision_summary.json', 'w') as f:
    json.dump(summary_data, f, indent=2)

print('\n' + '='*70)
print('VARIABLE PRECISION SWEEP COMPLETED SUCCESSFULLY!')
print('  results/variable_precision_summary.json')
print('  results/memory.csv')
print('  results/reconstruction.csv')
print('='*70)